# W87 / #46 — Multi-Modal Substrate Closure on Colab Pro

Closes the multi-modal substrate (vision + code + text) on a real open-weight VLM at Colab-Pro scale. Default: **`llava-hf/llava-1.5-7b-hf`** at bf16 on A100-40GB; falls back to **`HuggingFaceM4/idefics2-8b`** if LLaVA is unavailable.

The smaller local-CPU bench (`vikhyatk/moondream2`, 1.87 B params, fp32) is already validated and committed under `results/w87/multi_modal/` on `origin/main`. This notebook reproduces the closure at frontier scale.

## One-time setup

1. **Runtime → Change runtime type → A100 GPU** (L4 / V100 also work; T4 borderline for LLaVA-1.5-7B at bf16).
2. **HF Secret:** click the 🔑 key icon in the left sidebar → *+ Add new secret* → name = `hf_token`, value = `hf_xxxxxxxx`, toggle *Notebook access* **on**.  
   - LLaVA-1.5 is hosted publicly on HF; the token is used for rate-limit avoidance.
   - Idefics-2-8B is also public.
3. *Runtime → Run all*. Total wall-clock: ~5–7 min on A100 (~2 min model load + ~30 s bench + Drive copy).

## Output

* `multi_modal_v1_bench_report.json` written to `/content/w87_mm_<TS>/` and copied to Drive.
* The notebook prints a per-DoD-bullet PASS/FAIL summary from `verify_w87_multi_modal_audit_chain.py`.

## Anti-cheat

* All closure code lives in `coordpy/{multi_modal_payload_v1,vision_substrate_v1,code_substrate_v1,composed_multimodal_pipeline_v1}.py` on `origin/main`; this notebook only invokes the existing scripts.
* HF token never leaves Colab Secrets.
* Every CID re-derives offline.

In [ ]:
# --- 1. Environment probe + workdir ---
import os, sys, subprocess, json, datetime
RUN_TS = datetime.datetime.now(
    tz=datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = f'/content/w87_mm_{RUN_TS}'
os.makedirs(OUT_DIR, exist_ok=True)
print('RUN_TS:', RUN_TS)
print('OUT_DIR:', OUT_DIR)
subprocess.run(['nvidia-smi'], check=False)
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  mem={p.total_memory/(1024**3):.2f} GiB  sm={p.major}.{p.minor}')

In [ ]:
# --- 2. Install transformers + accelerate + pillow + einops + torchvision ---
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0',
    'accelerate>=0.34.0',
    'huggingface_hub>=0.24.0',
    'pillow', 'einops', 'torchvision', 'numpy>=1.24',
], check=True)
import transformers, accelerate, huggingface_hub, PIL, einops, torchvision
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)
print('huggingface_hub:', huggingface_hub.__version__)
print('PIL:', PIL.__version__)

In [ ]:
# --- 3. HF login via Colab Secrets ---
from google.colab import userdata  # type: ignore
hf_token = userdata.get('hf_token').strip()
assert hf_token.startswith('hf_'), (
    'Colab Secret `hf_token` must start with hf_. Add it via the key icon in the left sidebar.')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print('HF login OK; token len =', len(hf_token))

In [ ]:
# --- 4. Clone CoordPy ---
import shutil
shutil.rmtree('/content/CoordPy', ignore_errors=True)
subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/adotdong29/CoordPy.git',
    '/content/CoordPy',
], check=True)
os.chdir('/content/CoordPy')
print('HEAD =', subprocess.check_output(
    ['git', 'log', '-1', '--oneline'], text=True).strip())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
], check=True)
import coordpy
print('coordpy:', coordpy.__version__, 'SDK:', getattr(coordpy, 'SDK_VERSION', '?'))

In [ ]:
# --- 5. Run the multi-modal bench driver ---
VLM_MODEL = 'llava-hf/llava-1.5-7b-hf'  # alt: 'HuggingFaceM4/idefics2-8b'
CODE_MODEL = 'Qwen/Qwen2.5-Coder-1.5B'  # alt: 'distilgpt2'
BENCH_LOG = os.path.join(OUT_DIR, 'mm_run.log')
print('VLM_MODEL:', VLM_MODEL)
print('CODE_MODEL:', CODE_MODEL)
print('BENCH_LOG:', BENCH_LOG)
!cd /content/CoordPy && python scripts/run_w87_multi_modal_bench.py \
    --out-dir {OUT_DIR} \
    --vlm-model {VLM_MODEL} \
    --code-model {CODE_MODEL} \
    --require-real-vlm 2>&1 | tee {BENCH_LOG}
print()
print('=== OUT_DIR after driver ===')
!ls -la {OUT_DIR}

In [ ]:
# --- 6. Offline re-verifier ---
RPT = os.path.join(OUT_DIR, 'multi_modal_v1_bench_report.json')
!cd /content/CoordPy && python scripts/verify_w87_multi_modal_audit_chain.py --report {RPT}

In [ ]:
# --- 7. Print headline numbers ---
with open(RPT) as fh:
    rep = json.load(fh)
print('=== W87 #46 Multi-Modal Substrate closure headline ===')
for k in ('vlm_model_name', 'code_model_name',
          'vlm_loaded_real', 'code_loaded_real',
          'image_encoder_kind', 'code_encoder_kind',
          'image_hidden_shape', 'code_hidden_shape',
          'code_ast_n_functions', 'n_modalities',
          'cross_modality_root_cid', 'text_floor_fp32',
          'code_floor_fp32', 'image_floor_fp32',
          'multimodal_payload_for_three_modalities',
          'vision_adapter_loads_real_vlm_and_reads_hidden_state',
          'code_adapter_loads_real_codemodel_and_reads_hidden_state',
          'composed_pipeline_runs_on_team_with_at_least_two_modalities',
          'audit_chain_captures_all_modalities',
          'merkle_root_verifiable',
          'replay_byte_identity_per_modality_at_precision_floor',
          'bench_wall_seconds', 'bench_cid'):
    if k in rep:
        print(f'  {k}: {rep[k]}')

In [ ]:
# --- 8. Save to Drive + download zip ---
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=False)
DEST = f'/content/drive/MyDrive/coordpy_w87_multi_modal/w87_mm_{RUN_TS}'
os.makedirs(DEST, exist_ok=True)
subprocess.run(f'cp -r {OUT_DIR}/. {DEST}/', shell=True, check=False)
print('saved to Drive:', DEST)
subprocess.run(f'ls -la {DEST}', shell=True, check=False)
zip_path = f'/content/w87_mm_{RUN_TS}.zip'
subprocess.run(['zip', '-rq', zip_path, f'w87_mm_{RUN_TS}'],
               cwd='/content', check=False)
print('zip ready at', zip_path, f'({os.path.getsize(zip_path)/1024:.1f} KiB)')
from google.colab import files  # type: ignore
files.download(zip_path)